In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ─────────────────────────────────────────
#  CONFIG
# ─────────────────────────────────────────
DATA_DIR    = "data"
OUTPUT_DIR  = "data/processed"
DB_PATH     = f"{DATA_DIR}/logements/logements_rp2022.db"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 65)
print("🧹 ÉTAPE 3 — NETTOYAGE & FUSION DES DATASETS")
print("   JIRA : FRPT-4 — Nettoyage & Préparation")
print("=" * 65)


# ══════════════════════════════════════════════════════════════
#  UTILITAIRES
# ══════════════════════════════════════════════════════════════
def log_etape(titre):
    print(f"\n{'─'*65}")
    print(f"  {titre}")
    print(f"{'─'*65}")

def rapport_qualite(df, nom, avant=None):
    """Affiche un rapport qualité avant/après nettoyage"""
    print(f"\n  📋 {nom}")
    print(f"     Lignes    : {len(df):,}")
    if avant:
        print(f"     Supprimées: {avant - len(df):,} ({(avant-len(df))/avant*100:.1f}%)")
    manq = df.isnull().sum().sum()
    print(f"     Val. manq.: {manq:,}")
    doub = df.duplicated().sum()
    print(f"     Doublons  : {doub:,}")

def normaliser_codgeo(serie):
    """
    Normalise les codes INSEE commune :
    - Convertit en string
    - Supprime les espaces
    - Complète avec des zéros à gauche jusqu'à 5 caractères
    Ex: '1001' → '01001' | '75056' → '75056'
    """
    return (serie.astype(str)
                 .str.strip()
                 .str.replace(r'\.0$', '', regex=True)  # enlever .0 si numérique
                 .str.zfill(5))


# ══════════════════════════════════════════════════════════════
#  ÉTAPE 3.1 — NETTOYAGE DPE
# ══════════════════════════════════════════════════════════════
log_etape("3.1 — Chargement & Nettoyage DPE")

# Charger les 3 régions
dpe_parts = []
for code, nom in [("11","Île-de-France"), ("32","Hauts-de-France"), ("53","Bretagne")]:
    path = f"{DATA_DIR}/dpe/dpe_{code}.csv"
    if os.path.exists(path):
        df_part = pd.read_csv(path, dtype=str, low_memory=False)
        df_part["region"]     = nom
        df_part["region_code"]= code
        dpe_parts.append(df_part)
        print(f"  ✅ DPE {nom} : {len(df_part):,} lignes")

dpe = pd.concat(dpe_parts, ignore_index=True)
n_avant = len(dpe)
print(f"\n  Total DPE brut : {n_avant:,} lignes")

# ── Renommer colonne code INSEE ──────────────────────────────
for col in ["code_insee_commune_actualise", "code_insee_commune",
            "CODGEO", "codgeo", "code_commune"]:
    if col in dpe.columns:
        dpe = dpe.rename(columns={col: "CODGEO"})
        break

# ── Normaliser CODGEO ────────────────────────────────────────
dpe["CODGEO"] = normaliser_codgeo(dpe["CODGEO"])

# ── Supprimer lignes sans code INSEE ────────────────────────
dpe = dpe[dpe["CODGEO"].str.match(r'^\d{5}$', na=False)]

# ── Convertir types numériques ───────────────────────────────
cols_num_dpe = ["surface_habitable_logement", "consommation_energie_primaire",
                "annee_construction"]
for col in cols_num_dpe:
    if col in dpe.columns:
        dpe[col] = pd.to_numeric(dpe[col], errors="coerce")

# ── Filtrer valeurs aberrantes ───────────────────────────────
if "surface_habitable_logement" in dpe.columns:
    dpe = dpe[(dpe["surface_habitable_logement"].isna()) |
              ((dpe["surface_habitable_logement"] >= 9) &
               (dpe["surface_habitable_logement"] <= 500))]

if "annee_construction" in dpe.columns:
    dpe = dpe[(dpe["annee_construction"].isna()) |
              ((dpe["annee_construction"] >= 1800) &
               (dpe["annee_construction"] <= 2025))]

if "consommation_energie_primaire" in dpe.columns:
    dpe = dpe[(dpe["consommation_energie_primaire"].isna()) |
              ((dpe["consommation_energie_primaire"] >= 0) &
               (dpe["consommation_energie_primaire"] <= 2000))]

# ── Normaliser étiquette DPE ─────────────────────────────────
if "etiquette_dpe" in dpe.columns:
    dpe["etiquette_dpe"] = dpe["etiquette_dpe"].str.upper().str.strip()
    dpe = dpe[dpe["etiquette_dpe"].isin(["A","B","C","D","E","F","G"])]

# ── Supprimer doublons ───────────────────────────────────────
dpe = dpe.drop_duplicates()

# ── Colonnes utiles uniquement ───────────────────────────────
cols_dpe_garder = [c for c in [
    "CODGEO", "etiquette_dpe", "etiquette_ges", "type_batiment",
    "annee_construction", "surface_habitable_logement",
    "consommation_energie_primaire", "date_reception_dpe",
    "region", "region_code", "source"
] if c in dpe.columns]
dpe = dpe[cols_dpe_garder]

# ── Variable cible : est_passoire ────────────────────────────
dpe["est_passoire"] = dpe["etiquette_dpe"].isin(["F", "G"]).astype(int)

rapport_qualite(dpe, "DPE nettoyé", avant=n_avant)

# Sauvegarder DPE nettoyé
dpe.to_csv(f"{OUTPUT_DIR}/dpe_nettoye.csv", index=False, encoding="utf-8")
print(f"  💾 Sauvegardé → {OUTPUT_DIR}/dpe_nettoye.csv")


# ══════════════════════════════════════════════════════════════
#  ÉTAPE 3.2 — NETTOYAGE POPULATION
# ══════════════════════════════════════════════════════════════
log_etape("3.2 — Nettoyage Population")

pop = pd.read_csv(f"{DATA_DIR}/population/population_communes.csv",
                  dtype=str, low_memory=False)
n_avant = len(pop)

pop.columns = [c.strip().upper() for c in pop.columns]

# Normaliser CODGEO
for col in ["CODGEO", "CODE", "COM", "INSEE_COM"]:
    if col in pop.columns:
        pop = pop.rename(columns={col: "CODGEO"})
        break
pop["CODGEO"] = normaliser_codgeo(pop["CODGEO"])

# Trouver colonne population
col_pop = next((c for c in pop.columns
                if "POP" in c or c == "POPULATION"), None)
if col_pop:
    pop = pop.rename(columns={col_pop: "POPULATION"})
    pop["POPULATION"] = pd.to_numeric(pop["POPULATION"], errors="coerce")

# Trouver colonne nom commune
col_nom = next((c for c in pop.columns
                if "NOM" in c or "LIB" in c), None)
if col_nom:
    pop = pop.rename(columns={col_nom: "NOM_COMMUNE"})

# Colonnes à garder
cols_pop = [c for c in ["CODGEO", "NOM_COMMUNE", "POPULATION", "DEP", "REG"]
            if c in pop.columns]
pop = pop[cols_pop].drop_duplicates(subset=["CODGEO"])

# Supprimer communes sans code INSEE valide
pop = pop[pop["CODGEO"].str.match(r'^\d{5}$', na=False)]

# Supprimer population négative ou nulle
if "POPULATION" in pop.columns:
    pop = pop[pop["POPULATION"].isna() | (pop["POPULATION"] > 0)]

rapport_qualite(pop, "Population nettoyée", avant=n_avant)
pop.to_csv(f"{OUTPUT_DIR}/population_nettoyee.csv", index=False, encoding="utf-8")
print(f"  💾 Sauvegardé → {OUTPUT_DIR}/population_nettoyee.csv")


# ══════════════════════════════════════════════════════════════
#  ÉTAPE 3.3 — NETTOYAGE REVENUS
# ══════════════════════════════════════════════════════════════
log_etape("3.3 — Nettoyage Revenus")

rev = pd.read_csv(f"{DATA_DIR}/revenus/revenus_communes.csv",
                  dtype=str, low_memory=False)
n_avant = len(rev)

rev.columns = [c.strip().upper() for c in rev.columns]

# Normaliser CODGEO
for col in ["CODGEO", "CODE_COMMUNE", "COM", "CODE"]:
    if col in rev.columns:
        rev = rev.rename(columns={col: "CODGEO"})
        break
rev["CODGEO"] = normaliser_codgeo(rev["CODGEO"])

# Trouver et normaliser colonnes revenus
col_med = next((c for c in rev.columns if c.startswith("MED")), None)
col_tp  = next((c for c in rev.columns if c.startswith("TP")), None)
col_pauv = next((c for c in rev.columns if "PAUV" in c), None)

for col in [col_med, col_tp, col_pauv]:
    if col:
        rev[col] = pd.to_numeric(rev[col], errors="coerce")

# Renommer pour uniformiser
rename_rev = {}
if col_med:  rename_rev[col_med]  = "REVENU_MEDIAN"
if col_tp:   rename_rev[col_tp]   = "TAUX_PAUVRETE"
if col_pauv: rename_rev[col_pauv] = "MENAGES_PAUVRES"
rev = rev.rename(columns=rename_rev)

# Filtrer valeurs aberrantes revenus
if "REVENU_MEDIAN" in rev.columns:
    rev = rev[(rev["REVENU_MEDIAN"].isna()) |
              ((rev["REVENU_MEDIAN"] >= 5000) &
               (rev["REVENU_MEDIAN"] <= 100000))]

if "TAUX_PAUVRETE" in rev.columns:
    rev = rev[(rev["TAUX_PAUVRETE"].isna()) |
              ((rev["TAUX_PAUVRETE"] >= 0) &
               (rev["TAUX_PAUVRETE"] <= 60))]

# Colonnes à garder
cols_rev = [c for c in ["CODGEO", "NOM_COMMUNE", "REVENU_MEDIAN",
                         "TAUX_PAUVRETE", "MENAGES_PAUVRES",
                         "POPULATION", "DEP", "REG"]
            if c in rev.columns]
rev = rev[cols_rev].drop_duplicates(subset=["CODGEO"])
rev = rev[rev["CODGEO"].str.match(r'^\d{5}$', na=False)]

rapport_qualite(rev, "Revenus nettoyés", avant=n_avant)
rev.to_csv(f"{OUTPUT_DIR}/revenus_nettoyes.csv", index=False, encoding="utf-8")
print(f"  💾 Sauvegardé → {OUTPUT_DIR}/revenus_nettoyes.csv")


# ══════════════════════════════════════════════════════════════
#  ÉTAPE 3.4 — NETTOYAGE LOGEMENTS (depuis SQLite)
# ══════════════════════════════════════════════════════════════
log_etape("3.4 — Nettoyage Logements (lecture SQLite)")

conn = sqlite3.connect(DB_PATH)
log = pd.read_sql("SELECT * FROM logements_communes", conn)
conn.close()
n_avant = len(log)

log["CODGEO"] = normaliser_codgeo(log["CODGEO"].astype(str))

# Convertir colonnes numériques
cols_num_log = ["NB_LOGEMENTS", "NB_RP", "NB_LOGVAC", "NB_MAISON", "NB_APPART"]
for col in cols_num_log:
    if col in log.columns:
        log[col] = pd.to_numeric(log[col], errors="coerce")

# Filtrer incohérences
if "NB_LOGEMENTS" in log.columns and "NB_RP" in log.columns:
    # NB_RP ne peut pas dépasser NB_LOGEMENTS
    mask = (log["NB_RP"] > log["NB_LOGEMENTS"]) & log["NB_RP"].notna()
    log.loc[mask, "NB_RP"] = log.loc[mask, "NB_LOGEMENTS"]

# Valeurs négatives → NaN
for col in cols_num_log:
    if col in log.columns:
        log.loc[log[col] < 0, col] = np.nan

cols_log = [c for c in ["CODGEO", "DEP"] + cols_num_log
            if c in log.columns]
log = log[cols_log].drop_duplicates(subset=["CODGEO"])
log = log[log["CODGEO"].str.match(r'^\d{5}$', na=False)]

rapport_qualite(log, "Logements nettoyés", avant=n_avant)
log.to_csv(f"{OUTPUT_DIR}/logements_nettoyes.csv", index=False, encoding="utf-8")
print(f"  💾 Sauvegardé → {OUTPUT_DIR}/logements_nettoyes.csv")


# ══════════════════════════════════════════════════════════════
#  ÉTAPE 3.5 — AGRÉGATION DPE PAR COMMUNE
# ══════════════════════════════════════════════════════════════
log_etape("3.5 — Agrégation DPE par commune")

dpe_agg = dpe.groupby("CODGEO").agg(
    NB_DPE_TOTAL        = ("etiquette_dpe", "count"),
    NB_PASSOIRES        = ("est_passoire", "sum"),
    NB_CLASSE_A         = ("etiquette_dpe", lambda x: (x == "A").sum()),
    NB_CLASSE_B         = ("etiquette_dpe", lambda x: (x == "B").sum()),
    NB_CLASSE_C         = ("etiquette_dpe", lambda x: (x == "C").sum()),
    NB_CLASSE_D         = ("etiquette_dpe", lambda x: (x == "D").sum()),
    NB_CLASSE_E         = ("etiquette_dpe", lambda x: (x == "E").sum()),
    NB_CLASSE_F         = ("etiquette_dpe", lambda x: (x == "F").sum()),
    NB_CLASSE_G         = ("etiquette_dpe", lambda x: (x == "G").sum()),
    SURFACE_MOY         = ("surface_habitable_logement", "mean"),
    CONSO_ENERGIE_MOY   = ("consommation_energie_primaire", "mean"),
    ANNEE_CONSTRUCTION_MOY = ("annee_construction", "mean"),
    NB_MAISONS_DPE      = ("type_batiment",
                            lambda x: (x.str.lower() == "maison").sum()),
    NB_APPARTS_DPE      = ("type_batiment",
                            lambda x: (x.str.lower() == "appartement").sum()),
    REGION              = ("region", "first"),
    REGION_CODE         = ("region_code", "first"),
).reset_index()

# Taux de passoires
dpe_agg["TAUX_PASSOIRES"] = (
    dpe_agg["NB_PASSOIRES"] / dpe_agg["NB_DPE_TOTAL"] * 100
).round(2)

# Arrondir colonnes continues
for col in ["SURFACE_MOY", "CONSO_ENERGIE_MOY", "ANNEE_CONSTRUCTION_MOY"]:
    dpe_agg[col] = dpe_agg[col].round(1)

print(f"  ✅ DPE agrégé : {len(dpe_agg):,} communes")
print(f"     Colonnes   : {list(dpe_agg.columns)}")


# ══════════════════════════════════════════════════════════════
#  ÉTAPE 3.6 — FUSION DES 5 DATASETS
# ══════════════════════════════════════════════════════════════
log_etape("3.6 — Fusion des 5 datasets sur CODGEO")

print("  🔗 Jointures en cours...")

# Base : DPE agrégé par commune
df = dpe_agg.copy()
print(f"     Base DPE agrégé      : {len(df):,} communes")

# Jointure Population
df = df.merge(pop, on="CODGEO", how="left")
print(f"     + Population         : {df['POPULATION'].notna().sum():,} matchés")

# Jointure Revenus
df = df.merge(
    rev[["CODGEO", "REVENU_MEDIAN", "TAUX_PAUVRETE", "MENAGES_PAUVRES"]],
    on="CODGEO", how="left"
)
print(f"     + Revenus            : {df['REVENU_MEDIAN'].notna().sum():,} matchés")

# Jointure Logements
df = df.merge(
    log[["CODGEO", "NB_LOGEMENTS", "NB_RP", "NB_LOGVAC"]],
    on="CODGEO", how="left"
)
print(f"     + Logements          : {df['NB_LOGEMENTS'].notna().sum():,} matchés")

print(f"\n  ✅ Dataset fusionné : {len(df):,} communes × {df.shape[1]} colonnes")


# ══════════════════════════════════════════════════════════════
#  ÉTAPE 3.7 — CALCUL DES KPIs
# ══════════════════════════════════════════════════════════════
log_etape("3.7 — Calcul des KPIs")

# KPI 1 : Passoires pour 1000 habitants
df["PASSOIRES_POUR_1000_HAB"] = np.where(
    df["POPULATION"] > 0,
    (df["NB_PASSOIRES"] / df["POPULATION"] * 1000).round(2),
    np.nan
)

# KPI 2 : Taux couverture DPE (DPE / logements totaux)
df["TAUX_COUVERTURE_DPE"] = np.where(
    df["NB_LOGEMENTS"] > 0,
    (df["NB_DPE_TOTAL"] / df["NB_LOGEMENTS"] * 100).round(2),
    np.nan
)

# KPI 3 : Ratio passoires / résidences principales
df["RATIO_PASSOIRES_RP"] = np.where(
    df["NB_RP"] > 0,
    (df["NB_PASSOIRES"] / df["NB_RP"] * 100).round(2),
    np.nan
)

# KPI 4 : Part logements vacants
df["TAUX_VACANCE"] = np.where(
    df["NB_LOGEMENTS"] > 0,
    (df["NB_LOGVAC"] / df["NB_LOGEMENTS"] * 100).round(2),
    np.nan
)

print("  ✅ KPIs calculés :")
print("     - TAUX_PASSOIRES          (% logements F/G)")
print("     - PASSOIRES_POUR_1000_HAB (densité passoires)")
print("     - TAUX_COUVERTURE_DPE     (couverture DPE)")
print("     - RATIO_PASSOIRES_RP      (ratio / résidences principales)")
print("     - TAUX_VACANCE            (logements vacants)")


# ══════════════════════════════════════════════════════════════
#  ÉTAPE 3.8 — SCORE COMPOSITE DE PRIORITÉ
# ══════════════════════════════════════════════════════════════
log_etape("3.8 — Score composite de priorité")

def normaliser_minmax(serie):
    """Normalise une série entre 0 et 1 (Min-Max)"""
    s = pd.to_numeric(serie, errors="coerce")
    min_val = s.min()
    max_val = s.max()
    if max_val == min_val:
        return pd.Series(0, index=serie.index)
    return ((s - min_val) / (max_val - min_val)).round(4)

# Normaliser les 3 composantes
df["_norm_taux_passoires"]   = normaliser_minmax(df["TAUX_PASSOIRES"])
df["_norm_vulnerabilite_rev"] = 1 - normaliser_minmax(df["REVENU_MEDIAN"])
df["_norm_densite"]          = normaliser_minmax(df["PASSOIRES_POUR_1000_HAB"])

# Score composite pondéré (formule cahier des charges)
# Score = taux_passoires × 0.40 + vulnérabilité_revenus × 0.35 + densité × 0.25
df["SCORE_PRIORITE"] = (
    df["_norm_taux_passoires"]    * 0.40 +
    df["_norm_vulnerabilite_rev"] * 0.35 +
    df["_norm_densite"]           * 0.25
).round(4)

# Supprimer colonnes intermédiaires
df = df.drop(columns=[c for c in df.columns if c.startswith("_norm_")])

# Catégorie de priorité
df["CATEGORIE_PRIORITE"] = pd.cut(
    df["SCORE_PRIORITE"],
    bins=[0, 0.33, 0.66, 1.01],
    labels=["🟢 Faible", "🟡 Moyenne", "🔴 Haute"],
    include_lowest=True
)

# Rang de priorité
df["RANG_PRIORITE"] = df["SCORE_PRIORITE"].rank(
    ascending=False, method="min", na_option="bottom"
).astype("Int64")

print("  ✅ Score composite calculé")
print("     Formule : 0.40 × taux_passoires_norm")
print("             + 0.35 × (1 - revenu_norm)")
print("             + 0.25 × densité_passoires_norm")
print(f"\n  📊 Distribution des priorités :")
print(df["CATEGORIE_PRIORITE"].value_counts().to_string())


# ══════════════════════════════════════════════════════════════
#  ÉTAPE 3.9 — NETTOYAGE FINAL & EXPORT
# ══════════════════════════════════════════════════════════════
log_etape("3.9 — Nettoyage final & Export")

# Supprimer communes sans données DPE ni population
df_final = df[df["NB_DPE_TOTAL"] >= 5].copy()
print(f"  → Filtre min 5 DPE par commune : {len(df_final):,} communes retenues")

# Réordonner les colonnes logiquement
cols_ordre = [
    # Identification
    "CODGEO", "NOM_COMMUNE", "DEP", "REGION", "REGION_CODE",
    # DPE
    "NB_DPE_TOTAL", "NB_PASSOIRES", "TAUX_PASSOIRES",
    "NB_CLASSE_A","NB_CLASSE_B","NB_CLASSE_C","NB_CLASSE_D",
    "NB_CLASSE_E","NB_CLASSE_F","NB_CLASSE_G",
    "SURFACE_MOY", "CONSO_ENERGIE_MOY", "ANNEE_CONSTRUCTION_MOY",
    "NB_MAISONS_DPE", "NB_APPARTS_DPE",
    # Population
    "POPULATION",
    # Revenus
    "REVENU_MEDIAN", "TAUX_PAUVRETE", "MENAGES_PAUVRES",
    # Logements
    "NB_LOGEMENTS", "NB_RP", "NB_LOGVAC",
    # KPIs
    "PASSOIRES_POUR_1000_HAB", "TAUX_COUVERTURE_DPE",
    "RATIO_PASSOIRES_RP", "TAUX_VACANCE",
    # Score
    "SCORE_PRIORITE", "CATEGORIE_PRIORITE", "RANG_PRIORITE",
]
cols_finales = [c for c in cols_ordre if c in df_final.columns]
df_final = df_final[cols_finales]

# Export CSV principal (pour Power BI)
path_csv = f"{OUTPUT_DIR}/dataset_consolide.csv"
df_final.to_csv(path_csv, index=False, encoding="utf-8")
print(f"\n  💾 Dataset consolidé → {path_csv}")
print(f"     {len(df_final):,} communes × {df_final.shape[1]} colonnes")

# Export SQLite (dataset final en base)
conn_out = sqlite3.connect(f"{OUTPUT_DIR}/passoires_thermiques.db")
df_final.to_sql("dataset_consolide", conn_out, if_exists="replace", index=False)

# Vue Top 50 communes prioritaires
conn_out.execute("DROP VIEW IF EXISTS v_top50_prioritaires")
conn_out.execute("""
    CREATE VIEW v_top50_prioritaires AS
    SELECT CODGEO, NOM_COMMUNE, REGION, TAUX_PASSOIRES,
           REVENU_MEDIAN, PASSOIRES_POUR_1000_HAB,
           SCORE_PRIORITE, CATEGORIE_PRIORITE, RANG_PRIORITE
    FROM dataset_consolide
    WHERE SCORE_PRIORITE IS NOT NULL
    ORDER BY SCORE_PRIORITE DESC
    LIMIT 50
""")
conn_out.commit()
conn_out.close()
print(f"  💾 Base SQLite finale → {OUTPUT_DIR}/passoires_thermiques.db")


# ══════════════════════════════════════════════════════════════
#  RAPPORT FINAL
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("📊 RAPPORT FINAL — NETTOYAGE & FUSION")
print("=" * 65)

print(f"\n  📁 Fichiers produits :")
for f in os.listdir(OUTPUT_DIR):
    taille = os.path.getsize(f"{OUTPUT_DIR}/{f}") / (1024*1024)
    print(f"     ✅ {f:<40} {taille:.2f} Mo")

print(f"\n  📈 Statistiques dataset consolidé :")
print(f"     Communes analysées     : {len(df_final):,}")
print(f"     Régions couvertes      : {df_final['REGION'].nunique()}")
if "TAUX_PASSOIRES" in df_final.columns:
    tp = df_final["TAUX_PASSOIRES"]
    print(f"     Taux passoires moyen   : {tp.mean():.1f}%")
    print(f"     Taux passoires max     : {tp.max():.1f}%")
if "REVENU_MEDIAN" in df_final.columns:
    rm = df_final["REVENU_MEDIAN"]
    print(f"     Revenu médian moyen    : {rm.mean():,.0f} €")

print(f"\n  🏆 TOP 10 COMMUNES PRIORITAIRES :")
top10_cols = [c for c in ["NOM_COMMUNE","REGION","TAUX_PASSOIRES",
                           "REVENU_MEDIAN","SCORE_PRIORITE","RANG_PRIORITE"]
              if c in df_final.columns]
print(df_final.sort_values("SCORE_PRIORITE", ascending=False)
              [top10_cols].head(10).to_string(index=False))

print(f"\n{'='*65}")


🧹 ÉTAPE 3 — NETTOYAGE & FUSION DES DATASETS
   JIRA : FRPT-4 — Nettoyage & Préparation

─────────────────────────────────────────────────────────────────
  3.1 — Chargement & Nettoyage DPE
─────────────────────────────────────────────────────────────────
  ✅ DPE Île-de-France : 5,000 lignes
  ✅ DPE Hauts-de-France : 5,000 lignes
  ✅ DPE Bretagne : 5,000 lignes

  Total DPE brut : 15,000 lignes

  📋 DPE nettoyé
     Lignes    : 15,000
     Supprimées: 0 (0.0%)
     Val. manq.: 0
     Doublons  : 0
  💾 Sauvegardé → data/processed/dpe_nettoye.csv

─────────────────────────────────────────────────────────────────
  3.2 — Nettoyage Population
─────────────────────────────────────────────────────────────────

  📋 Population nettoyée
     Lignes    : 6,250
     Supprimées: 0 (0.0%)
     Val. manq.: 0
     Doublons  : 0
  💾 Sauvegardé → data/processed/population_nettoyee.csv

─────────────────────────────────────────────────────────────────
  3.3 — Nettoyage Revenus
───────────────────────────